# 06 — Final Labeled Lip Dataset Builder (Manifest-Only)

สร้าง training manifest จากข้อมูล final โดย **ไม่ copy รูป mouth ซ้ำ** เพื่อประหยัดพื้นที่ Drive D

Input:
- `dataset/final_alignment/video001/final_speaker_transcript_auto.csv`
- `dataset/mouth_crops_final/video001/mouth_metadata.csv`

Output:
- `dataset/final_dataset/video001/segment_qa.csv`
- `dataset/final_dataset/video001/manifest_train_ready.csv`
- `dataset/final_dataset/video001/accepted_segments.csv`
- `dataset/final_dataset/video001/rejected_segments.csv`

## Cell 1 — Path

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

ALIGNMENT_CSV = PROJECT_ROOT / "dataset" / "final_alignment" / VIDEO_ID / "final_speaker_transcript_auto.csv"
MOUTH_METADATA_CSV = PROJECT_ROOT / "dataset" / "mouth_crops_final" / VIDEO_ID / "mouth_metadata.csv"
OUTPUT_ROOT = PROJECT_ROOT / "dataset" / "final_dataset" / VIDEO_ID

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("ALIGNMENT_CSV     :", ALIGNMENT_CSV)
print("MOUTH_METADATA_CSV:", MOUTH_METADATA_CSV)
print("OUTPUT_ROOT       :", OUTPUT_ROOT)

if not ALIGNMENT_CSV.exists():
    raise FileNotFoundError(ALIGNMENT_CSV)
if not MOUTH_METADATA_CSV.exists():
    raise FileNotFoundError(MOUTH_METADATA_CSV)

## Cell 2 — Load

In [ ]:
import pandas as pd
import numpy as np
import re

align_df = pd.read_csv(ALIGNMENT_CSV)
mouth_df = pd.read_csv(MOUTH_METADATA_CSV)

if mouth_df["accepted"].dtype == object:
    mouth_df["accepted"] = mouth_df["accepted"].astype(str).str.lower().isin(["true","1","yes"])

mouth_df = mouth_df[mouth_df["accepted"] == True].copy()

for c in ["frame_index","speaker_id"]:
    mouth_df[c] = mouth_df[c].astype(int)

for c in ["segment_id","speaker_id","start_frame","end_frame"]:
    align_df[c] = align_df[c].astype(int)

print("AUTO segments      :", len(align_df))
print("Accepted mouth rows:", len(mouth_df))
print("Speakers           :", sorted(align_df["speaker_id"].unique().tolist()))
display(align_df.head())

## Cell 3 — QA Thresholds

In [ ]:
MIN_DURATION_SEC = 0.40
MAX_DURATION_SEC = 8.00
MIN_FRAMES = 10
MIN_COVERAGE = 0.85
MIN_CONTINUITY = 0.90
MIN_TEXT_LENGTH = 2
MAX_TEXT_LENGTH = 180

print("Thresholds ready")

## Cell 4 — Clean Text

In [ ]:
def clean_text(text):
    return re.sub(r"\s+", " ", str(text).strip())

def suspicious_text(text):
    text = clean_text(text)
    thai_chars = len(re.findall(r"[ก-๙]", text))
    alpha_chars = len(re.findall(r"[A-Za-zก-๙]", text))
    if alpha_chars == 0:
        return True
    return (thai_chars / alpha_chars) < 0.50

align_df["text_clean"] = align_df["text"].apply(clean_text)
align_df["text_length"] = align_df["text_clean"].str.len()
align_df["suspicious_text"] = align_df["text_clean"].apply(suspicious_text)

display(align_df[["segment_id","text_clean","text_length","suspicious_text"]].head(30))

## Cell 5 — Segment QA

In [ ]:
qa_rows = []

for _, seg in align_df.iterrows():
    segment_id = int(seg["segment_id"])
    speaker_id = int(seg["speaker_id"])
    start_frame = int(seg["start_frame"])
    end_frame = int(seg["end_frame"])
    start_time = float(seg["start_time"])
    end_time = float(seg["end_time"])
    duration = end_time - start_time
    text = str(seg["text_clean"])

    expected_frames = max(1, end_frame - start_frame + 1)

    matched = mouth_df[
        (mouth_df["speaker_id"] == speaker_id) &
        (mouth_df["frame_index"] >= start_frame) &
        (mouth_df["frame_index"] <= end_frame)
    ].copy().sort_values("frame_index")

    frame_indices = matched["frame_index"].drop_duplicates().astype(int).tolist()
    observed_frames = len(frame_indices)
    coverage = observed_frames / expected_frames if expected_frames else 0.0

    if observed_frames <= 1:
        continuity = 0.0
    else:
        diffs = np.diff(frame_indices)
        continuity = float(np.mean(diffs <= 2))

    reasons = []
    if duration < MIN_DURATION_SEC: reasons.append("too_short")
    if duration > MAX_DURATION_SEC: reasons.append("too_long")
    if observed_frames < MIN_FRAMES: reasons.append("too_few_frames")
    if coverage < MIN_COVERAGE: reasons.append("low_coverage")
    if continuity < MIN_CONTINUITY: reasons.append("low_continuity")
    if len(text) < MIN_TEXT_LENGTH: reasons.append("text_too_short")
    if len(text) > MAX_TEXT_LENGTH: reasons.append("text_too_long")
    if bool(seg["suspicious_text"]): reasons.append("suspicious_text")

    qa_rows.append({
        "video_id": VIDEO_ID,
        "segment_id": segment_id,
        "speaker_id": speaker_id,
        "start_time": start_time,
        "end_time": end_time,
        "duration": round(duration, 3),
        "start_frame": start_frame,
        "end_frame": end_frame,
        "expected_frames": expected_frames,
        "observed_frames": observed_frames,
        "coverage": round(coverage, 4),
        "continuity": round(continuity, 4),
        "text": text,
        "status": "ACCEPT" if not reasons else "REJECT",
        "reject_reason": "|".join(reasons),
    })

qa_df = pd.DataFrame(qa_rows)

print(qa_df["status"].value_counts())
display(qa_df.head(30))

## Cell 6 — Reject Summary

In [ ]:
reject_df = qa_df[qa_df["status"] == "REJECT"].copy()

reason_counts = {}
for value in reject_df["reject_reason"].fillna(""):
    for reason in str(value).split("|"):
        if reason:
            reason_counts[reason] = reason_counts.get(reason, 0) + 1

reason_summary = pd.DataFrame(
    list(reason_counts.items()),
    columns=["reason","count"]
).sort_values("count", ascending=False)

display(reason_summary)
display(reject_df[[
    "segment_id","speaker_id","duration","observed_frames",
    "coverage","continuity","text","reject_reason"
]].head(60))

## Cell 7 — Build Manifest Without Copying Images

In [ ]:
manifest_rows = []

accepted_df = qa_df[qa_df["status"] == "ACCEPT"].copy()

for _, seg in accepted_df.iterrows():
    speaker_id = int(seg["speaker_id"])
    start_frame = int(seg["start_frame"])
    end_frame = int(seg["end_frame"])
    segment_id = int(seg["segment_id"])

    matched = mouth_df[
        (mouth_df["speaker_id"] == speaker_id) &
        (mouth_df["frame_index"] >= start_frame) &
        (mouth_df["frame_index"] <= end_frame)
    ].copy().sort_values("frame_index")

    frame_paths = matched["mouth_file"].astype(str).tolist()
    frame_indices = matched["frame_index"].astype(int).tolist()

    if not frame_paths:
        continue

    clip_id = f"{VIDEO_ID}_seg{segment_id:06d}_spk{speaker_id:03d}"

    manifest_rows.append({
        "clip_id": clip_id,
        "video_id": VIDEO_ID,
        "segment_id": segment_id,
        "speaker_id": speaker_id,
        "start_time": float(seg["start_time"]),
        "end_time": float(seg["end_time"]),
        "duration": float(seg["duration"]),
        "start_frame": start_frame,
        "end_frame": end_frame,
        "num_frames": len(frame_paths),
        "coverage": float(seg["coverage"]),
        "continuity": float(seg["continuity"]),
        "text": seg["text"],
        "frame_paths": "|".join(frame_paths),
        "frame_indices": "|".join(map(str, frame_indices)),
        "label_status": "weak_label_auto",
    })

manifest_df = pd.DataFrame(manifest_rows)

print("Train-ready clips:", len(manifest_df))
display(manifest_df.head(20))

## Cell 8 — Dataset Summary

In [ ]:
if len(manifest_df) == 0:
    raise RuntimeError("ไม่มี clip ที่ผ่าน QA")

print("Clips by speaker:")
display(manifest_df["speaker_id"].value_counts())

print("\nDuration summary:")
display(manifest_df["duration"].describe())

print("\nFrame count summary:")
display(manifest_df["num_frames"].describe())

print("\nTotal training duration (minutes):",
      round(manifest_df["duration"].sum()/60, 2))

## Cell 9 — Verify Paths

In [ ]:
from pathlib import Path

missing = []
checked = 0

for _, row in manifest_df.iterrows():
    for rel in str(row["frame_paths"]).split("|"):
        checked += 1
        p = Path(rel)
        full = p if p.is_absolute() else PROJECT_ROOT / p

        if not full.exists():
            missing.append(str(full))
            if len(missing) >= 50:
                break

    if len(missing) >= 50:
        break

print("Checked paths:", checked)
print("Missing paths:", len(missing))

if missing:
    for p in missing[:10]:
        print(p)
else:
    print("All checked paths exist")

## Cell 10 — Save

In [ ]:
QA_CSV = OUTPUT_ROOT / "segment_qa.csv"
MANIFEST_CSV = OUTPUT_ROOT / "manifest_train_ready.csv"
ACCEPTED_CSV = OUTPUT_ROOT / "accepted_segments.csv"
REJECTED_CSV = OUTPUT_ROOT / "rejected_segments.csv"

qa_df.to_csv(QA_CSV, index=False, encoding="utf-8-sig")
manifest_df.to_csv(MANIFEST_CSV, index=False, encoding="utf-8-sig")
qa_df[qa_df["status"]=="ACCEPT"].to_csv(ACCEPTED_CSV, index=False, encoding="utf-8-sig")
qa_df[qa_df["status"]=="REJECT"].to_csv(REJECTED_CSV, index=False, encoding="utf-8-sig")

print("Saved:")
print(" -", QA_CSV)
print(" -", MANIFEST_CSV)
print(" -", ACCEPTED_CSV)
print(" -", REJECTED_CSV)

## Cell 11 — Preview Samples

In [ ]:
sample_n = min(20, len(manifest_df))
sample_df = manifest_df.sample(
    n=sample_n,
    random_state=42
).sort_values("start_time")

display(sample_df[[
    "clip_id","speaker_id","duration","num_frames",
    "coverage","continuity","text"
]])

# ขั้นต่อไป — 07 Dataset Split

หลัง Notebook นี้ผ่าน จะได้:
`dataset/final_dataset/video001/manifest_train_ready.csv`

จากนั้นทำ train/val/test split แล้วค่อยเริ่ม baseline training